# AccentShift â€” Seed-VC Fine-Tuning on Kaggle

**Before running:**
1. Settings (right panel) â†’ **Accelerator: GPU T4 x2** or P100
2. Settings â†’ **Internet: On**
3. Add-ons â†’ **Secrets** â†’ add `TELEGRAM_TOKEN` and `TELEGRAM_CHAT_ID` (optional but recommended)
4. Run All

Expected total time: ~2â€“3 hours on T4, ~3â€“5 hours on P100.

In [ ]:
# Cell 1: Verify GPU
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# Cell 2: Telegram setup (reads from Kaggle Secrets)
# Add-ons â†’ Secrets â†’ add TELEGRAM_TOKEN and TELEGRAM_CHAT_ID before running
import os
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['TELEGRAM_TOKEN'] = secrets.get_secret('TELEGRAM_TOKEN')
    os.environ['TELEGRAM_CHAT_ID'] = secrets.get_secret('TELEGRAM_CHAT_ID')
    print('Telegram secrets loaded OK')
except Exception as e:
    print(f'Telegram secrets not found ({e}) â€” training will run without notifications')

In [ ]:
# Cell 2b: Test Telegram BEFORE training — verify notifications work
import os, urllib.request, urllib.parse

token = os.environ.get('TELEGRAM_TOKEN', '').strip()
chat_id = os.environ.get('TELEGRAM_CHAT_ID', '').strip()

if not token or not chat_id:
    print('No Telegram secrets found. Run Cell 2 first, or skip.')
else:
    url = f'https://api.telegram.org/bot{token}/sendMessage'
    msg = 'AccentShift Kaggle — Telegram test OK! Training will notify here.'
    data = urllib.parse.urlencode({'chat_id': chat_id, 'text': msg}).encode()
    try:
        urllib.request.urlopen(url, data=data, timeout=10)
        print('Telegram test PASSED — check your phone!')
    except Exception as e:
        print(f'Telegram FAILED: {e}')
        print('Check: token has no spaces, chat_id is correct integer')


In [ ]:
%%bash
# Cell 3: Clone repo (idempotent — pulls latest if already cloned)
if [ ! -d /kaggle/working/AccentShift ]; then
    echo '=== Cloning AccentShift repo ==='
    git clone --depth 1 https://github.com/nischal2805/AccentShift.git /kaggle/working/AccentShift
else
    echo '=== Repo exists, pulling latest div branch ==='
    cd /kaggle/working/AccentShift
    git fetch --depth 1 origin div
    git checkout div
    git reset --hard origin/div
fi
cd /kaggle/working/AccentShift
git checkout div 2>/dev/null || true
echo "Branch: $(git branch --show-current)"
echo "Latest commit: $(git log --oneline -1)"
echo '=== Repo ready ==='


In [ ]:
%%bash
# Cell 4: Install pipeline deps (torch already on Kaggle)
echo '=== Installing deps ==='
pip install -q \n
    "librosa>=0.10.2" soundfile pyloudnorm pyworld silero-vad \n
    "transformers==4.48.0" jiwer speechbrain scikit-learn joblib pyyaml click \n
    huggingface-hub tqdm hydra-core omegaconf einops munch \n
    accelerate pydub tensorboard gdown 2>&1 | tail -5
echo '=== Deps installed ==='


In [ ]:
%%bash
# Cell 5: Clone Seed-VC + Amphion + install seed-vc requirements
cd /kaggle/working/AccentShift/backend
mkdir -p third_party

echo '=== Cloning Seed-VC ==='
if [ ! -d third_party/seed-vc ]; then
    git clone --depth 1 --filter=blob:none --single-branch \
        https://github.com/Plachtaa/seed-vc.git third_party/seed-vc
    echo 'Seed-VC cloned'
else
    echo 'Seed-VC already present'
fi

echo '=== Installing Seed-VC deps ==='
pip install -q -r third_party/seed-vc/requirements.txt --no-deps 2>/dev/null || true
echo 'Seed-VC deps done'

echo '=== Cloning Amphion ==='
if [ ! -d third_party/Amphion ]; then
    git clone --depth 1 --filter=blob:none --single-branch \
        https://github.com/open-mmlab/Amphion.git third_party/Amphion
    echo 'Amphion cloned'
else
    echo 'Amphion already present'
fi

echo '=== third_party ready ==='

In [ ]:
%%bash
# Cell 6: Download L2-Arctic v5.0 from Google Drive (~7GB)
echo '=== Downloading L2-Arctic dataset ==='
mkdir -p /kaggle/working/AccentShift/backend/data/l2arctic_raw
ZIP=/kaggle/working/AccentShift/backend/data/l2arctic_raw/l2arctic_v5.zip

if [ ! -f "$ZIP" ]; then
    gdown 'https://drive.google.com/uc?id=1ciCw_ttbw7a9r7d5DZzTJwoZq5rQB3TA' -O "$ZIP"
else
    echo 'ZIP already downloaded, skipping'
fi
echo "ZIP size: $(du -sh $ZIP)"

In [ ]:
%%bash
# Cell 7: Unzip + organize WAVs (skips if already done)
cd /kaggle/working/AccentShift/backend
RAW=data/l2arctic_raw

# Count existing WAVs across all accent dirs
EXISTING=$(find data/finetune -name '*.wav' 2>/dev/null | wc -l)
echo "Existing organized WAVs: $EXISTING"

if [ "$EXISTING" -gt 1000 ]; then
    echo 'Dataset already organized, skipping unzip+organize.'
    echo '=== WAV counts per accent ==='
    for d in data/finetune/*/; do
        [ -d "$d" ] || continue
        echo "  $(basename $d): $(find $d -name '*.wav' | wc -l) WAVs"
    done
    exit 0
fi

echo '=== Unzipping main archive ==='
unzip -o -q $RAW/l2arctic_v5.zip -d $RAW
echo 'Main zip extracted'

echo '=== Freeing space: deleting main zip ==='
rm -f $RAW/l2arctic_v5.zip
df -h /kaggle/working

echo '=== Unzipping per-speaker archives (deleting each after extract) ==='
for spk_zip in $RAW/*.zip; do
    [ -f "$spk_zip" ] || continue
    spk=$(basename "$spk_zip" .zip)
    echo "  $spk..."
    unzip -o -q "$spk_zip" -d $RAW
    rm -f "$spk_zip"
done
echo 'All speakers extracted'
df -h /kaggle/working

echo '=== Organizing WAVs by accent ==='
python3 - << 'EOF'
from pathlib import Path
import shutil

SPEAKER_ACCENT = {
    'ASI':'indian_english','RRBI':'indian_english','SVBI':'indian_english','TNI':'indian_english',
    'BWC':'chinese_english','LXC':'chinese_english','NCC':'chinese_english','TXHC':'chinese_english',
    'HJK':'korean_english','HKK':'korean_english','YDCK':'korean_english','YKWK':'korean_english',
    'HQTV':'vietnamese_english','PNV':'vietnamese_english','THV':'vietnamese_english','TLV':'vietnamese_english',
    'EBVS':'spanish_english','ERMS':'spanish_english','MBMPS':'spanish_english','NJS':'spanish_english',
    'ABA':'arabic_english','SKA':'arabic_english','YBAA':'arabic_english','ZHAA':'arabic_english',
}

raw = Path('/kaggle/working/AccentShift/backend/data/l2arctic_raw')
ft = Path('/kaggle/working/AccentShift/backend/data/finetune')

for spk, accent in SPEAKER_ACCENT.items():
    out = ft / accent
    out.mkdir(parents=True, exist_ok=True)
    found = 0
    for candidate in [raw/spk/'wav', raw/spk, raw/'l2arctic_v5'/spk/'wav', raw/'l2arctic_v5'/spk]:
        if candidate.is_dir():
            for wav in candidate.rglob('*.wav'):
                dst = out / wav.name
                if not dst.exists():
                    shutil.copy2(wav, dst)
                found += 1
            break
    print(f'  {spk} -> {accent} ({found} WAVs)')

print('')
for d in sorted(ft.iterdir()):
    if d.is_dir():
        count = len(list(d.glob('*.wav')))
        print(f'  {d.name}: {count} WAVs')
EOF


In [ ]:
# Cell 8: TRAIN — ~1.5-2 hours on T4 x2 (multi-GPU via accelerate)
import subprocess, sys, os, shutil

os.chdir('/kaggle/working/AccentShift/backend')
os.environ['HF_HOME'] = '/kaggle/working/AccentShift/backend/.hf_cache'

import torch
n_gpus = torch.cuda.device_count()
tg_token = os.environ.get('TELEGRAM_TOKEN', '').strip()
tg_chat = os.environ.get('TELEGRAM_CHAT_ID', '').strip()
print('GPUs available:', n_gpus)
print('TELEGRAM_TOKEN set:', bool(tg_token))
print('TELEGRAM_CHAT_ID set:', bool(tg_chat))
print('')

use_multi_gpu = n_gpus > 1 and shutil.which('accelerate') is not None

if use_multi_gpu:
    eff_batch = 8 * n_gpus
    print('Multi-GPU mode:', n_gpus, 'x T4 (effective batch =', eff_batch, ')')
    cmd = [
        'accelerate', 'launch',
        '--num_processes', str(n_gpus),
        '--mixed_precision', 'fp16',
        'scripts/finetune_style.py',
        '--accent', 'all',
        '--steps', '15000',
        '--batch-size', '8',
        '--save-every', '1000',
        '--num-workers', '2',
        '--mixed-precision', 'fp16',
    ]
else:
    print('Single-GPU mode (n_gpus=' + str(n_gpus) + ')')
    cmd = [
        sys.executable, 'scripts/finetune_style.py',
        '--accent', 'all',
        '--steps', '15000',
        '--batch-size', '8',
        '--save-every', '1000',
        '--num-workers', '2',
        '--mixed-precision', 'fp16',
    ]

print('CMD:', ' '.join(cmd))
import datetime
print('=== Starting at', datetime.datetime.now().strftime('%H:%M:%S'), '===')
print('')

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd='/kaggle/working/AccentShift/backend',
    env=os.environ.copy(),
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print('=== Exit code:', proc.returncode, '===')
print('=== Finished at', datetime.datetime.now().strftime('%H:%M:%S'), '===')


In [ ]:
# Cell 8b: RESUME from checkpoint - run this instead of Cell 8 if checkpoint exists
import subprocess, sys, os, shutil, datetime, pathlib

BACKEND = '/kaggle/working/AccentShift/backend'
os.chdir(BACKEND)
os.environ['HF_HOME'] = BACKEND + '/.hf_cache'

OLD_CKPT = BACKEND + '/third_party/seed-vc/runs/all_ft/CFM_epoch_00009_step_08000.pth'

# Pre-stage finetune_all BEFORE launching accelerate - avoids multi-GPU sentinel race
merged = pathlib.Path(BACKEND) / 'data' / 'finetune_all'
sentinel = pathlib.Path(BACKEND) / 'data' / 'finetune_all.ready'
inprogress = pathlib.Path(BACKEND) / 'data' / 'finetune_all.inprogress'
inprogress.unlink(missing_ok=True)
sentinel.unlink(missing_ok=True)
if merged.exists():
    shutil.rmtree(merged)
merged.mkdir(parents=True)
ft_root = pathlib.Path(BACKEND) / 'data' / 'finetune'
total = 0
for accent_dir in sorted(ft_root.iterdir()):
    if not accent_dir.is_dir(): continue
    for wav in accent_dir.rglob('*.wav'):
        dst = merged / f'{accent_dir.name}__{wav.name}'
        try:
            dst.symlink_to(wav.resolve())
        except (OSError, NotImplementedError):
            shutil.copy2(wav, dst)
        total += 1
sentinel.touch()
print(f'Pre-staged {total} WAVs -> {merged}')

import torch
n_gpus = torch.cuda.device_count()
print('GPUs available:', n_gpus)
print('Resuming from:', OLD_CKPT)
print('')

use_multi_gpu = n_gpus > 1 and shutil.which('accelerate') is not None

base_args = [
    '--data-dir', str(merged),
    '--run-name', 'all_ft',
    '--steps', '10000',
    '--batch-size', '8',
    '--save-every', '500',
    '--num-workers', '2',
    '--mixed-precision', 'fp16',
    '--pretrained-cfm', OLD_CKPT,
]

if use_multi_gpu:
    cmd = ['accelerate', 'launch', '--num_processes', str(n_gpus),
           '--mixed_precision', 'fp16', 'scripts/finetune_style.py'] + base_args
else:
    cmd = [sys.executable, 'scripts/finetune_style.py'] + base_args

print('CMD:', ' '.join(cmd))
print('=== Resuming at', datetime.datetime.now().strftime('%H:%M:%S'), '===')
print('')

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    cwd=BACKEND,
    env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'
=== Exit code: {proc.returncode} ===')
print('=== Finished at', datetime.datetime.now().strftime('%H:%M:%S'), '===')


In [ ]:
%%bash
# Cell 9: Verify checkpoints (checks both new and old locations)
NEW=/kaggle/working/AccentShift/backend/runs
OLD=/kaggle/working/AccentShift/backend/third_party/seed-vc/runs

echo '=== New location (backend/runs/) ==='
find "$NEW" -name '*.pth' -exec du -sh {} \; 2>/dev/null || echo '  (empty)'

echo ''
echo '=== Old location (seed-vc/runs/) ==='
find "$OLD" -name '*.pth' -exec du -sh {} \; 2>/dev/null || echo '  (empty)'

echo ''
NEW_COUNT=$(find "$NEW" -name '*.pth' 2>/dev/null | wc -l)
OLD_COUNT=$(find "$OLD" -name '*.pth' 2>/dev/null | wc -l)
echo "New location: $NEW_COUNT checkpoint(s)"
echo "Old location: $OLD_COUNT checkpoint(s)"


In [ ]:
import shutil, os, glob
# Cell 10: Zip checkpoints — collects from both new and old locations

NEW_RUNS = '/kaggle/working/AccentShift/backend/runs/'
OLD_RUNS = '/kaggle/working/AccentShift/backend/third_party/seed-vc/runs/'
MERGE_DIR = '/kaggle/working/all_checkpoints'
OUT_ZIP = '/kaggle/working/seedvc_finetuned'

os.makedirs(MERGE_DIR, exist_ok=True)

total = 0
for runs_dir in [NEW_RUNS, OLD_RUNS]:
    for pth in glob.glob(f'{runs_dir}/**/*.pth', recursive=True):
        rel = os.path.relpath(pth, runs_dir)
        dst = os.path.join(MERGE_DIR, rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if not os.path.exists(dst):
            shutil.copy2(pth, dst)
            total += 1

print(f'Collected {total} checkpoint(s) into {MERGE_DIR}')

if total == 0:
    print('WARNING: no .pth files found in either location!')
else:
    shutil.make_archive(OUT_ZIP, 'zip', MERGE_DIR)
    size = os.path.getsize(OUT_ZIP + '.zip') / (1024**2)
    print(f'Zipped: {OUT_ZIP}.zip ({size:.0f} MB)')
    print('Download via Files panel on the right ->')
